# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

> **This version runs with current LangChain/LangSmith and uses OpenAI's GPT Luna (`gpt-6-luna`)** in place of the old `OpenAI()` completion model. `HuggingFaceHub`, `run_on_dataset` and `RunEvalConfig` no longer exist, so:
> - the Hugging Face models run **locally** with `transformers` (no Hugging Face token needed);
> - the experiments run with `client.evaluate(...)`;
> - the `embedding_distance` and `string_distance` evaluators are re-implemented with the same definitions, plus an LLM-as-judge evaluator on GPT Luna.
>
> As the Intro asks, a **second, different dataset** (DialogSum, dialogue summaries) is evaluated at the end.

Keys come from a `.env` file next to the notebook: `OPENAI_API_KEY`, `LANGSMITH_API_KEY` (or `LANGCHAIN_API_KEY`) and `LANGSMITH_TRACING=true`.

In [ ]:
# %pip install -qU langsmith langchain-openai "datasets<4" transformers sentencepiece torch rapidfuzz pandas python-dotenv

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
LANGCHAIN_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
assert OPENAI_API_KEY and LANGCHAIN_API_KEY, "Add OPENAI_API_KEY and LANGSMITH_API_KEY to your .env file"

In [2]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_PROJECT", "lab-langsmith-summarization")

'lab-langsmith-summarization'

In [3]:
#Importing Client from Langsmith
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

### Create Dataset


In [4]:
from datasets import load_dataset
# "abisee/cnn_dailymail" is the same CNN/DailyMail data as "ccdv/cnn_dailymail", stored as plain files (no loading script)
cnn_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

In [5]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [6]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [7]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [8]:
#Get just a few news to test
MAX_NEWS=10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [9]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [10]:
import datetime

In [11]:
input_key=['article']
output_key=['highlights']

NAME_DATASET=f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H-%M-%S')}"

In [12]:
#This creates the dataset in LangSmith with the content in sample_cnn (a new name each run, so re-running is safe)
dataset = client.upload_dataframe(
    df=sample_cnn.to_pandas()[["article", "highlights"]],
    input_keys=input_key,
    output_keys=output_key,
    name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
    data_type="kv"
)
print("Created dataset:", NAME_DATASET)

Created dataset: Summarize_dataset_2026-11-09 16-03-54


In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base T5 model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

`HuggingFaceHub` (the hosted inference API) was removed from LangChain, and most of these models are no longer served there, so both run **locally** with `transformers`. The first run downloads them (about 0.9 GB each).

- **Base model:** `t5-base`, the same base model as the original notebook.
- **Fine-tuned model:** `sshleifer/distilbart-cnn-6-6`, fine-tuned on the training split of this same CNN/DailyMail dataset. The original `flax-community/t5-base-cnn-dm` only ships Flax weights, which don't run in a standard PyTorch setup.

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# transformers 5 removed the "summarization" pipeline, so we call the seq2seq models directly.
# load_summarizer returns a function with the same interface the pipeline had.
def load_summarizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).eval()
    max_input = min(tokenizer.model_max_length, 1024)   # T5: 512 tokens, BART: 1024

    def summarize(text, max_length=180, min_length=30, **_):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input)
        with torch.no_grad():
            ids = model.generate(**enc, max_length=max_length, min_length=min_length, num_beams=4, do_sample=False)
        return [{"summary_text": tokenizer.decode(ids[0], skip_special_tokens=True)}]
    return summarize

summarizer_base = load_summarizer("t5-base")
summarizer_finetuned = load_summarizer("sshleifer/distilbart-cnn-6-6")


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

C:\Users\Roberto\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub\models--t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

C:\Users\Roberto\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

... (9 outputs left out)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  460MB            

model.safetensors: downloading bytes:           |  0.00B            

In [14]:
PREFIX = "Summarize this news:\n"

def _article(inputs: dict) -> str:
    return inputs["article"].removeprefix(PREFIX)

# Each target receives a dataset example's inputs and returns {"output": summary}
def t5_base(inputs: dict) -> dict:
    # T5 was trained with the task prefix "summarize: "
    out = summarizer_base("summarize: " + _article(inputs), max_length=180, min_length=30, truncation=True, do_sample=False)
    return {"output": out[0]["summary_text"]}

def bart_finetuned(inputs: dict) -> dict:
    out = summarizer_finetuned(_article(inputs), max_length=180, min_length=30, truncation=True, do_sample=False)
    return {"output": out[0]["summary_text"]}

print(bart_finetuned(sample_cnn[0])["output"])

 The Palestinian Authority officially becomes the 123rd member of the International Criminal Court . The move gives the court jurisdiction over alleged crimes in Palestinian territories . Israel and the U.S. opposed the Palestinians' efforts to join the body .


## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.


`RunEvalConfig(evaluators=["embedding_distance", "string_distance"])` is re-implemented below with the same definitions:
- **embedding_distance**: cosine distance between the embeddings of the model summary and the reference highlights (OpenAI embeddings, as LangChain's default). **Lower is better.**
- **string_distance**: normalized Jaro-Winkler distance between the two texts (LangChain's default string distance). **Lower is better.** It only compares characters, so it punishes good summaries worded differently.
- **summary_quality** (new): GPT Luna as a judge, checking that the summary is faithful to the article and covers the reference's key points. **Higher is better**, with the judge's explanation saved as a comment.

In [15]:
import numpy as np
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from pydantic import BaseModel, Field
from rapidfuzz.distance import JaroWinkler

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def embedding_distance(outputs: dict, reference_outputs: dict) -> dict:
    a, b = embeddings.embed_documents([outputs["output"], reference_outputs["highlights"]])
    a, b = np.array(a), np.array(b)
    return {"key": "embedding_distance", "score": float(1 - a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))}

def string_distance(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "string_distance",
            "score": float(JaroWinkler.normalized_distance(outputs["output"], reference_outputs["highlights"]))}

In [16]:
LUNA_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

class SummaryGrade(BaseModel):
    explanation: str = Field(description="Short reasoning for the score")
    faithful: bool = Field(description="True if the summary contains no information that is not in the article")
    coverage: int = Field(description="1-5: how many of the reference's key points the summary covers (5 = all)")

judge = ChatOpenAI(model=LUNA_MODEL, reasoning_effort="low").with_structured_output(SummaryGrade)

def summary_quality(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    g = judge.invoke(
        "You are grading a news summary. Compare the SUMMARY with the ARTICLE (for faithfulness) and with the "
        "REFERENCE highlights written by a human (for coverage of the key points).\n\n"
        f"ARTICLE:\n{inputs['article'][:6000]}\n\nREFERENCE:\n{reference_outputs['highlights']}\n\nSUMMARY:\n{outputs['output']}"
    )
    # 1.0 = faithful and covers everything; an unfaithful summary scores 0
    score = (g.coverage - 1) / 4 if g.faithful else 0.0
    return {"key": "summary_quality", "score": score, "comment": g.explanation}

#We are using the two classic LangSmith string evaluators plus an LLM-as-judge.
evaluators = [embedding_distance, string_distance, summary_quality]

### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

In [17]:
def run_experiment(target, prefix, data=None, **metadata):
    return client.evaluate(
        target,
        data=data or NAME_DATASET,
        evaluators=evaluators,
        experiment_prefix=prefix,
        metadata=metadata,
        max_concurrency=1,   # local models run one at a time
    )

base_t5_results = run_experiment(t5_base, "T5-BASE", model="t5-base")

View the evaluation results for experiment: 'T5-BASE-fef7f9a0' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8e25c274-155f-42ac-95c2-34b57511890f/compare?selectedSessions=b67142ea-0931-43e6-8044-c66635be3256




0it [00:00, ?it/s]

In [18]:
finetuned_results = run_experiment(bart_finetuned, "BART-FineTuned-CNN", model="sshleifer/distilbart-cnn-6-6")

View the evaluation results for experiment: 'BART-FineTuned-CNN-f858a539' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8e25c274-155f-42ac-95c2-34b57511890f/compare?selectedSessions=3156fa3e-f9eb-4bbc-87fa-1e7b1e41e415




0it [00:00, ?it/s]

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model? Here: **GPT Luna**.

In [19]:
luna = ChatOpenAI(model=LUNA_MODEL, reasoning_effort="low")

def luna_summarizer(inputs: dict) -> dict:
    msg = luna.invoke(inputs["article"] + "\n\nWrite the summary as 3-4 short highlight sentences.")
    return {"output": msg.content}

luna_results = run_experiment(luna_summarizer, "GPT-Luna", model=LUNA_MODEL)

View the evaluation results for experiment: 'GPT-Luna-3d428776' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/8e25c274-155f-42ac-95c2-34b57511890f/compare?selectedSessions=f262ad4b-920f-4b1a-8dd1-0e74aca0f853




0it [00:00, ?it/s]

### Comparing the three models

In [20]:
import pandas as pd

def scores(results, name):
    df = results.to_pandas()
    cols = [c for c in df.columns if c.startswith("feedback.")]
    s = df[cols].astype(float).mean().rename(lambda c: c.replace("feedback.", ""))
    s["avg_latency_s"] = df["execution_time"].mean() if "execution_time" in df else np.nan
    s["avg_summary_words"] = df["outputs.output"].str.split().str.len().mean()
    return s.rename(name)

cnn_comparison = pd.concat([
    scores(base_t5_results, "T5-base"),
    scores(finetuned_results, "DistilBART fine-tuned on CNN"),
    scores(luna_results, "GPT Luna"),
], axis=1).T.round(3)
cnn_comparison

,embedding_distance,string_distance,summary_quality,avg_latency_s,avg_summary_words
T5-base,0.375,0.276,0.200,18.820,40.5
DistilBART fine-tuned on CNN,0.319,0.284,0.425,9.768,43.4
GPT Luna,0.255,0.344,0.675,1.729,72.7


<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.

## A different dataset: DialogSum
The Intro asks to pick different sets of data and re-run the evaluation. **DialogSum** contains everyday conversations with human-written summaries, which is a very different kind of text from news articles. The same three models and evaluators run on 10 dialogues.

In [21]:
dialog = load_dataset("knkarthick/dialogsum", split="test").select(range(10))

NAME_DIALOG = f"DialogSum_{datetime.datetime.now().strftime('%Y-%m-%d %H-%M-%S')}"
client.upload_dataframe(
    df=pd.DataFrame({"article": [PREFIX + d for d in dialog["dialogue"]], "highlights": dialog["summary"]}),
    input_keys=["article"], output_keys=["highlights"],
    name=NAME_DIALOG, description="DialogSum dialogue summarization", data_type="kv",
)
print(dialog[0]["dialogue"][:500], "\n---\nReference:", dialog[0]["summary"])

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

C:\Users\Roberto\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub\datasets--knkarthick--dialogsum. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv: reconstructing file:   0%|          |  0.00B / 11.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to in 
---
Reference: Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.


In [22]:
dialog_t5 = run_experiment(t5_base, "DIALOG-T5-BASE", data=NAME_DIALOG, model="t5-base")
dialog_bart = run_experiment(bart_finetuned, "DIALOG-BART-FineTuned-CNN", data=NAME_DIALOG, model="sshleifer/distilbart-cnn-6-6")
dialog_luna = run_experiment(luna_summarizer, "DIALOG-GPT-Luna", data=NAME_DIALOG, model=LUNA_MODEL)

dialog_comparison = pd.concat([
    scores(dialog_t5, "T5-base"),
    scores(dialog_bart, "DistilBART fine-tuned on CNN"),
    scores(dialog_luna, "GPT Luna"),
], axis=1).T.round(3)
dialog_comparison

View the evaluation results for experiment: 'DIALOG-T5-BASE-6078cc06' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/354cbd0b-720d-4aea-807e-2336c6c3cd7d/compare?selectedSessions=db1390c7-9299-4733-8534-0e1ea8969769




0it [00:00, ?it/s]

View the evaluation results for experiment: 'DIALOG-BART-FineTuned-CNN-df0e2729' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/354cbd0b-720d-4aea-807e-2336c6c3cd7d/compare?selectedSessions=0f85cc27-41b9-433c-8721-90d73de1f926




0it [00:00, ?it/s]

View the evaluation results for experiment: 'DIALOG-GPT-Luna-9d8a3e92' at:
https://smith.langchain.com/o/7e73e74c-b144-45dd-82b5-51928b0c32c0/datasets/354cbd0b-720d-4aea-807e-2336c6c3cd7d/compare?selectedSessions=2c87c9e2-7ba7-4df1-8951-8595fcc34638




0it [00:00, ?it/s]

,embedding_distance,string_distance,summary_quality,avg_latency_s,avg_summary_words
T5-base,0.353,0.311,0.350,15.828,46.9
DistilBART fine-tuned on CNN,0.400,0.334,0.225,7.455,43.3
GPT Luna,0.357,0.349,0.650,1.555,44.8


In [23]:
# Same models, two datasets: how much does each model's quality depend on the data?
pd.concat({"CNN news": cnn_comparison["summary_quality"], "DialogSum": dialog_comparison["summary_quality"]}, axis=1)

,CNN news,DialogSum
T5-base,0.200,0.350
DistilBART fine-tuned on CNN,0.425,0.225
GPT Luna,0.675,0.650


In LangSmith, open **Datasets & Experiments**, pick a dataset, tick its three experiments and click **Compare** to read the summaries side by side. Click any `summary_quality` score to read the judge's explanation.

## What did you learn?
- **A LangSmith evaluation has three parts:** a dataset of inputs with reference outputs, a target (the model or chain) and evaluators. Once these exist, comparing a new model is one more `evaluate()` call, and every run is kept as an experiment that can be compared later.
- **Each metric measures something different.** `string_distance` only compares characters, so it penalises good summaries that use different words. `embedding_distance` compares meaning and is a better automatic signal. An LLM judge is the only one of the three that can check faithfulness (no invented facts) and explain its score, but it costs API calls and should be spot-checked.
- **Fine-tuning matters for the data it was trained on.** The model fine-tuned on CNN/DailyMail writes news-style highlights, which the reference-based metrics reward on news. The DialogSum run shows how far that advantage carries over to a different kind of text.
- **Prompts and input format matter.** T5 expects the `summarize:` prefix it was trained with, and a general model like GPT Luna needs to be told the length and style of summary expected.
- **Quality isn't the only dimension.** LangSmith also records latency and token cost for each run. The local models are free but slower on a CPU; GPT Luna costs a little per call. That trade-off belongs in the comparison too.